In [ ]:
%pip install -q -r requirements.txt --no-cache-dir --force-reinstall

In [ ]:
!pip freeze | grep boto
!pip freeze | grep agentcore

In [26]:
import importlib, helpers.utils
importlib.reload(helpers.utils)
from helpers.utils import create_agentcore_runtime_execution_role, delete_agentcore_runtime_execution_role, WEATHER_AGENT_ROLE_NAME

weather_agent_role_arn = create_agentcore_runtime_execution_role(WEATHER_AGENT_ROLE_NAME)
# execution_role_arn_mcp = create_agentcore_runtime_execution_role(AWS_DOCS_ROLE_NAME)
# delete_agentcore_runtime_execution_role(WEATHER_AGENT_ROLE_NAME)


✅ Created IAM role: weather-agent-role-us-east-1
Role ARN: arn:aws:iam::281024298475:role/weather-agent-role-us-east-1
✅ Created policy: a2a-with-agentcore-demo-policy-us-east-1
✅ Attached policy to role
Policy ARN: arn:aws:iam::281024298475:policy/a2a-with-agentcore-demo-policy-us-east-1


In [19]:
import os
import json
import requests
import boto3
import time
from boto3.session import Session
from strands.tools import tool

boto_session = Session()
region = boto_session.region_name

In [27]:
from bedrock_agentcore_starter_toolkit import Runtime

weather_agent_runtime = Runtime()
weather_agent_name="weather_agent"

# Configure the deployment
response_weather_agent_configure = weather_agent_runtime.configure(
    entrypoint="agents/weather_agent.py",
    execution_role=weather_agent_role_arn,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=weather_agent_name,
    protocol="A2A",
)

print("Configuration completed:", response_weather_agent_configure)

Entrypoint parsed: file=/Users/antonaws/dev/tmp/a2a-workshop/agents/weather_agent.py, bedrock_agentcore_name=weather_agent
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: weather_agent


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Using existing Dockerfile: /Users/antonaws/dev/tmp/a2a-workshop/Dockerfile

Generated .dockerignore: /Users/antonaws/dev/tmp/a2a-workshop/.dockerignore
Keeping 'weather_agent' as default agent
Bedrock AgentCore configured: /Users/antonaws/dev/tmp/a2a-workshop/.bedrock_agentcore.yaml


Configuration completed: config_path=PosixPath('/Users/antonaws/dev/tmp/a2a-workshop/.bedrock_agentcore.yaml') dockerfile_path=PosixPath('/Users/antonaws/dev/tmp/a2a-workshop/Dockerfile') dockerignore_path=PosixPath('/Users/antonaws/dev/tmp/a2a-workshop/.dockerignore') runtime='None' runtime_type=None region='us-east-1' account_id='281024298475' execution_role='arn:aws:iam::281024298475:role/weather-agent-role-us-east-1' ecr_repository=None auto_create_ecr=True s3_path=None auto_create_s3=False memory_id=None network_mode='PUBLIC' network_subnets=None network_security_groups=None network_vpc_id=None


In [28]:
response_weather_agent_launch = weather_agent_runtime.launch()
print("Launch completed:", response_weather_agent_launch.agent_arn)

weather_agent_arn = response_weather_agent_launch.agent_arn

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'weather_agent' to account 281024298475 (us-east-1)
Generated image tag: 20260408-222604-556
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: weather_agent
ECR repository available: 281024298475.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-weather_agent
Using execution role from config: arn:aws:iam::281024298475:role/weather-agent-role-us-east-1
Preparing CodeBuild project and uploading source...


✅ Reusing existing ECR repository: 281024298475.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-weather_agent


Getting or creating CodeBuild execution role for agent: weather_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-425e274c35
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-425e274c35
Creating IAM role: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-425e274c35
✓ Role created: arn:aws:iam::281024298475:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-425e274c35
Attaching inline policy: CodeBuildExecutionPolicy to role: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-425e274c35
✓ Policy attached: CodeBuildExecutionPolicy
Waiting for IAM role propagation...
CodeBuild execution role creation complete: arn:aws:iam::281024298475:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-425e274c35
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: weather_agent/source.zip
Created CodeBuild project: bedrock-agentcore-weather_agent-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBui

Launch completed: arn:aws:bedrock-agentcore:us-east-1:281024298475:runtime/weather_agent-UxkAwgDlQe


In [29]:
status_response = weather_agent_runtime.status()
status = status_response.endpoint["status"]

print(f"Final status: {status}")

Retrieved Bedrock AgentCore status for: weather_agent


Final status: READY


In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime_blogs = Runtime()
aws_blogs_agent_name="aws_blog_assistant"

# Configure the deployment
response_aws_blogs_agent = agentcore_runtime_blogs.configure(
    entrypoint="agents/strands_aws_blogs_news.py",
    execution_role=execution_role_arn_blogs,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=aws_blogs_agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="A2A"
)

print("Configuration completed:", response_aws_blogs_agent)

In [ ]:
launch_result_blog = agentcore_runtime_blogs.launch()
print("Launch completed:", launch_result_blog.agent_arn)

blog_agent_arn = launch_result_blog.agent_arn

In [ ]:
status_response = agentcore_runtime_blogs.status()
status = status_response.endpoint["status"]

print(f"Final status: {status}")

In [ ]:
MCP_AGENT_ID = launch_result_mcp.agent_id
MCP_AGENT_ARN = launch_result_mcp.agent_arn
MCP_AGENT_NAME = aws_docs_agent_name

BLOG_AGENT_ID = launch_result_blog.agent_id
BLOG_AGENT_ARN = launch_result_blog.agent_arn
BLOG_AGENT_NAME = aws_blogs_agent_name

COGNITO_CLIENT_ID = cognito_config.get("client_id")
COGNITO_SECRET = cognito_config.get("client_secret")
DISCOVERY_URL = cognito_config.get("discovery_url")

%store MCP_AGENT_ID
%store MCP_AGENT_ARN
%store MCP_AGENT_NAME
%store BLOG_AGENT_ID
%store BLOG_AGENT_ARN
%store BLOG_AGENT_NAME
%store COGNITO_CLIENT_ID
%store COGNITO_SECRET
%store DISCOVERY_URL

In [ ]:
from helpers.utils import put_ssm_parameter, SSM_DOCS_AGENT_ARN, SSM_BLOGS_AGENT_ARN

put_ssm_parameter(SSM_DOCS_AGENT_ARN, MCP_AGENT_ARN)
put_ssm_parameter(SSM_BLOGS_AGENT_ARN, BLOG_AGENT_ARN)

In [ ]:
bearer_token = reauthenticate_user(
    cognito_config.get("client_id"), 
    cognito_config.get("client_secret")
)
print(bearer_token)

In [ ]:
import logging
from uuid import uuid4
from urllib.parse import quote

logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

def fetch_agent_card(agent_arn):
    # URL encode the agent ARN
    escaped_agent_arn = quote(agent_arn, safe='')

    # Construct the URL
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/.well-known/agent-card.json"
    logger.info(url)
    # Generate a unique session ID
    session_id = str(uuid4())
    logger.info(f"Generated session ID: {session_id}")

    # Set headers
    headers = {
        'Accept': '*/*',
        'Authorization': f'Bearer {bearer_token}',
        'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
        'X-Amzn-Trace-Id': f'aws_docs_assistant_{session_id}'
    }

    try:
        # Make the request
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # Parse and pretty print JSON
        agent_card = response.json()
        logger.info(json.dumps(agent_card, indent=2))

        return agent_card

    except requests.exceptions.RequestException as e:
        logger.error(f"Error fetching agent card: {e}")
        return None

In [ ]:
fetch_agent_card(docs_agent_arn)

In [ ]:
fetch_agent_card(blog_agent_arn)

In [ ]:
import asyncio
import logging
import os
from uuid import uuid4

import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.types import Message, Part, Role, TextPart

logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

DEFAULT_TIMEOUT = 300  # set request timeout to 5 minutes

def format_agent_response(response):
    """Extract and format agent response for human readability."""
    # Get the main response text from artifacts
    if response.artifacts and len(response.artifacts) > 0:
        artifact = response.artifacts[0]
        if artifact.parts and len(artifact.parts) > 0:
            return artifact.parts[0].root.text
    
    # Fallback: concatenate all agent messages from history
    agent_messages = [
        msg.parts[0].root.text 
        for msg in response.history 
        if msg.role.value == 'agent' and msg.parts
    ]
    return ''.join(agent_messages)


def create_message(*, role: Role = Role.user, text: str) -> Message:
    return Message(
        kind="message",
        role=role,
        parts=[Part(TextPart(kind="text", text=text))],
        message_id=uuid4().hex,
    )

async def send_sync_message(agent_arn, message: str):
    # Get runtime URL from environment variable
    escaped_agent_arn = quote(agent_arn, safe='')

    # Construct the URL
    runtime_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/"
    
    # Generate a unique session ID
    session_id = str(uuid4())
    print(f"Generated session ID: {session_id}")

    # Add authentication headers for AgentCore
    headers = {"Authorization": f"Bearer {bearer_token}",
              'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id}
        
    async with httpx.AsyncClient(timeout=DEFAULT_TIMEOUT, headers=headers) as httpx_client:
        # Get agent card from the runtime URL
        resolver = A2ACardResolver(httpx_client=httpx_client, base_url=runtime_url)
        agent_card = await resolver.get_agent_card()
        print(agent_card)

        # Agent card contains the correct URL (same as runtime_url in this case)
        # No manual override needed - this is the path-based mounting pattern

        # Create client using factory
        config = ClientConfig(
            httpx_client=httpx_client,
            streaming=False,  # Use non-streaming mode for sync response
        )
        factory = ClientFactory(config)
        client = factory.create(agent_card)

        # Create and send message
        msg = create_message(text=message)

        # With streaming=False, this will yield exactly one result
        async for event in client.send_message(msg):
            if isinstance(event, Message):
                logger.info(event.model_dump_json(exclude_none=True, indent=2))
                return event
            elif isinstance(event, tuple) and len(event) == 2:
                # (Task, UpdateEvent) tuple
                task, update_event = event
                logger.info(f"Task: {task.model_dump_json(exclude_none=True, indent=2)}")
                if update_event:
                    logger.info(f"Update: {update_event.model_dump_json(exclude_none=True, indent=2)}")
                return task
            else:
                # Fallback for other response types
                logger.info(f"Response: {str(event)}")
                return event

In [ ]:
result = await send_sync_message(docs_agent_arn, "what is DynamoDB")
formatted_output = format_agent_response(result)
print(formatted_output)

In [ ]:
result = await send_sync_message(blog_agent_arn, "Give me the latest published blog for Bedrock AgentCore?")
formatted_output = format_agent_response(result)
print(formatted_output)

In [ ]:
def format_agent_trace(response):
    """Format agent response as a readable trace of calls."""
    print("=" * 60)
    print("🔍 AGENT EXECUTION TRACE")
    print("=" * 60)
    
    # Context info
    print(f"📋 Context ID: {response.context_id}")
    print(f"🆔 Task ID: {response.id}")
    print(f"📊 Status: {response.status.state.value}")
    print(f"⏰ Completed: {response.status.timestamp}")
    print()
    
    # Trace through history
    print("🔄 EXECUTION FLOW:")
    print("-" * 40)
    
    for i, msg in enumerate(response.history, 1):
        role_icon = "👤" if msg.role.value == "user" else "🤖"
        text = msg.parts[0].root.text if msg.parts else "[No content]"
        
        # Truncate long messages for trace view
        if len(text) > 80:
            text = text[:77] + "..."
            
        print(f"{i:2d}. {role_icon} {msg.role.value.upper()}: {text}")
    
    print()
    print("✅ FINAL RESULT:")
    print("-" * 40)
    
    # Final artifact
    if response.artifacts:
        final_text = response.artifacts[0].parts[0].root.text
        print(final_text[:200] + "..." if len(final_text) > 200 else final_text)
    
    print("=" * 60)